# Preprocessing & Data Tools
# 数据处理与预处理工具

This tutorial covers PipelineTS data loading, preprocessing, and utility functions:
本教程介绍 PipelineTS 的数据加载、预处理和工具函数：

1. **Built-in datasets / 内置数据集**: Load various time series datasets
1. **内置数据集**：加载各种时序数据集

2. **Data generator / 数据生成器**: Generate synthetic time series data
2. **数据生成器**：生成合成时序数据

3. **Data scalers / 数据缩放器 (Scaler)**: Data normalization and inverse transformation
3. **数据缩放器**：数据标准化与反标准化

4. **Sequence splitting / 序列分割**: Convert time series to supervised learning format
4. **序列分割**：将时序数据分割为监督学习格式

5. **Evaluation metrics / 评估指标**: Built-in prediction evaluation metrics
5. **评估指标**：内置各种预测评估指标

6. **Interval prediction accuracy / 区间预测准确率**: quantile_acc
6. **区间预测准确率**：quantile_acc

## 1. Built-in Datasets
## 1. 内置数据集

PipelineTS provides multiple built-in datasets for quick experimentation.
PipelineTS 提供多个内置数据集，适合快速实验。

In [ ]:
from PipelineTS.dataset import (
    LoadElectricDataSets,
    LoadMessagesSentHourDataSets,
    LoadMessagesSentDataSets,
    LoadWebSales,
    LoadSupermarketIncoming,
    BuiltInSeriesData
)

datasets = {
    'Electric': LoadElectricDataSets,
    'MessagesSentHour': LoadMessagesSentHourDataSets,
    'MessagesSent': LoadMessagesSentDataSets,
    'WebSales': LoadWebSales,
    'SupermarketIncoming': LoadSupermarketIncoming,
}

for name, loader in datasets.items():
    df = loader()
    print(f"{name}: shape={df.shape}, columns={df.columns.tolist()}")

## 2. Data Generator
## 2. 数据生成器

Use `DataGenerator` to generate synthetic time series data, suitable for testing and benchmarking.
使用 `DataGenerator` 生成合成时序数据，适合测试和基准测试。

In [ ]:
from PipelineTS.dataset import DataGenerator

# Generate synthetic time series / 生成合成时间序列
synthetic_data = DataGenerator(n=200)
print(type(synthetic_data))
synthetic_data

## 3. Data Scalers (Scaler)
## 3. 数据缩放器 (Scaler)

PipelineTS provides a unified `Scaler` interface supporting 4 scaling methods.
PipelineTS 提供统一的 `Scaler` 接口，支持 4 种缩放方式。

In [ ]:
import numpy as np
from PipelineTS.preprocessing import Scaler

# Prepare data / 准备数据
X = np.random.randn(100, 1)

# Supported scaler types / 支持的缩放器类型: 'min_max', 'standard', 'quantile', 'gauss_rank'
for scaler_name in ['min_max', 'standard', 'quantile', 'gauss_rank']:
    scaler = Scaler(scaler_name)
    transformed = scaler.fit_transform(X)
    recovered = scaler.inverse_transform(transformed)
    print(f"{scaler_name:12s} | Scaled range / 缩放后范围: [{transformed.min():.3f}, {transformed.max():.3f}] | "
          f"Recovery error / 恢复误差: {np.abs(X - recovered).max():.2e}")

## 4. Sequence Splitting
## 4. 序列分割

Convert 1D/multi-dimensional time series into supervised learning format (X, y).
将一维/多维时间序列转换为监督学习格式 (X, y)。

In [ ]:
from PipelineTS.spinesTS.preprocessing import (
    split_series,
    train_test_split_ts,
    lag_splits,
    split_series_multivariate
)

# Univariate sequence splitting / 单变量序列分割
series = np.sin(np.linspace(0, 4 * np.pi, 100))
X, y = split_series(series, in_features=10, out_features=5)
print(f"split_series: X.shape={X.shape}, y.shape={y.shape}")

# Time-series train/test split (preserves temporal order)
# 时序训练/测试集分割（保持时序顺序）
X_train, X_test, y_train, y_test = train_test_split_ts(X, y, train_size=0.8)
print(f"train: X={X_train.shape}, y={y_train.shape}")
print(f"test:  X={X_test.shape}, y={y_test.shape}")

In [ ]:
# Multivariate sequence splitting (3D data)
# 多变量序列分割（3D 数据）
multi_series = np.random.randn(100, 3).astype(np.float32)
X_mv, y_mv = split_series_multivariate(multi_series, in_features=10, out_features=5)
print(f"split_series_multivariate: X.shape={X_mv.shape}, y.shape={y_mv.shape}")
print(f"  Dimensions: (samples, timesteps, variables)")
print(f"  维度含义: (样本数, 时间步, 变量数)")

## 5. Evaluation Metrics
## 5. 评估指标

In [ ]:
from PipelineTS.spinesTS.metrics import mae, mse, rmse, wmape

y_true = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y_pred = np.array([1.1, 2.2, 2.8, 4.1, 5.3])

print(f"MAE:   {mae(y_true, y_pred):.4f}")
print(f"MSE:   {mse(y_true, y_pred):.4f}")
print(f"RMSE:  {rmse(y_true, y_pred):.4f}")
print(f"WMAPE: {wmape(y_true, y_pred):.4f}")

## 6. Interval Prediction Accuracy (quantile_acc)
## 6. 区间预测准确率 (quantile_acc)

Evaluate the coverage rate of prediction intervals.
评估预测区间的覆盖率。

In [ ]:
from PipelineTS.metrics import quantile_acc

y_true = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
lower  = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
upper  = np.array([1.5, 2.5, 3.5, 4.5, 5.5])

acc = quantile_acc(y_true, lower, upper)
print(f"Interval coverage rate / 区间覆盖率: {acc:.2%}")

## Summary / 总结

| Feature / 功能 | API |
|---|---|
| Built-in datasets / 内置数据集 | `LoadElectricDataSets()`, `LoadWebSales()` etc. |
| Data generation / 数据生成 | `DataGenerator(n=100)` |
| Data scaling / 数据缩放 | `Scaler('min_max')` / `Scaler('standard')` |
| Sequence splitting / 序列分割 | `split_series()`, `split_series_multivariate()` |
| Evaluation metrics / 评估指标 | `mae()`, `rmse()`, `wmape()` |
| Interval accuracy / 区间准确率 | `quantile_acc(yt, lower, upper)` |